# Importing libraries

In [14]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from keras.models import Sequential
from keras.layers import Dense,Dropout
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV,KFold,train_test_split
from scikeras.wrappers import KerasClassifier

# Data Collections

In [2]:
diabetes_df = pd.read_csv("diabetes.csv")
diabetes_df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


# Data Understanding

In [3]:
diabetes_df.shape

(768, 9)

In [4]:
diabetes_df.isna().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [5]:
diabetes_df.dtypes

Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object

In [6]:
diabetes_df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


# Data Preparations

In [7]:
x = diabetes_df.drop(labels = "Outcome",axis= 1)
y = diabetes_df["Outcome"]


In [8]:
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size=0.20,random_state=12,shuffle=True,stratify=y)

In [9]:
X_train.shape,y_train.shape

((614, 8), (614,))

In [10]:
X_test.shape,y_test.shape

((154, 8), (154,))

In [11]:
standard_scale = StandardScaler()

In [12]:
x_train = standard_scale.fit_transform(X_train)
x_test  = standard_scale.transform(X_test)


# Hyperparameter Turning

In [13]:
def create_model(learning_rate=0.001,dropout_rate=0.0,activation_function='relu', init='glorot_uniform',neuron1=8, neuron2=4):
    model = Sequential()
    model.add(Dense(neuron1,input_shape=(8,),kernel_initializer=init,activation=activation_function))
    model.add(Dropout(dropout_rate))
    model.add(Dense(neuron2,kernel_initializer=init,activation=activation_function))    
    model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='sigmoid'))
    
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='binary_crossentropy',optimizer=optimizer,metrics=['accuracy'])
    return model

In [15]:
model = KerasClassifier(model = create_model,verbose= 2) 

In [ ]:
param_grid = {'model__learning_rate': [0.001, 0.01], 
              'model__dropout_rate': [0.0, 0.2],
              'model__activation_function': ['relu', 'tanh'],
              'model__init': ['glorot_uniform', 'he_uniform'],
              'model__neuron1': [8, 16],
              'model__neuron2': [4, 8],
              'batch_size': [10, 20],
              'epochs': [10, 50]
             }

In [ ]:
kfold = KFold(n_splits=3,shuffle=True,random_state=22)
grid = GridSearchCV(estimator=model,param_grid=param_grid,cv=kfold,n_jobs=-1)

In [ ]:
grid_result = grid.fit(x_train, y_train)

In [ ]:
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']

for mean, std, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, std, param))

In [ ]:
results_df = pd.DataFrame(grid_result.cv_results_)
results_df